# SemEval Data preparation

In [1]:
import os
import huggingface_hub
import pandas as pd
import datasets
from datasets import Dataset, DatasetDict, load_dataset
from dotenv import load_dotenv
from typing import Union, List, Optional

/Users/paul/miniconda3/envs/DeepImpact/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_API_KEY")
huggingface_hub.login(token=hf_token)

In [3]:
train = '/Users/paul/Desktop/GitHub repositories/StanceDetector/data_processing/semeval_data/train_utf8.csv'
test = '/Users/paul/Desktop/GitHub repositories/StanceDetector/data_processing/semeval_data/test_utf8.csv'

In [4]:
df_train = pd.read_csv(train)
df_test = pd.read_csv(test)

In [5]:
df_train.head()

,Tweet,Target,Stance,Opinion Towards,Sentiment
0,"@tedcruz And, #HandOverTheServer she wiped cle...",Hillary Clinton,AGAINST,1. The tweet explicitly expresses opinion abo...,neg
1,Hillary is our best choice if we truly want to...,Hillary Clinton,FAVOR,1. The tweet explicitly expresses opinion abo...,pos
2,@TheView I think our country is ready for a fe...,Hillary Clinton,AGAINST,1. The tweet explicitly expresses opinion abo...,neg
3,I just gave an unhealthy amount of my hard-ear...,Hillary Clinton,AGAINST,1. The tweet explicitly expresses opinion abo...,neg
4,@PortiaABoulger Thank you for adding me to you...,Hillary Clinton,NONE,3. The tweet is not explicitly expressing opi...,pos


Check that the percentages for each label match those reported in `Saif Mohammad, Svetlana Kiritchenko, Parinaz Sobhani, Xiaodan Zhu, and Colin Cherry. 2016. SemEval-2016 task 6: Detecting stance in tweets.`

In [6]:
df_train.groupby(['Target'])['Stance'].value_counts(normalize=True)

Target                            Stance 
Atheism                           AGAINST    0.592593
                                  NONE       0.228070
                                  FAVOR      0.179337
Climate Change is a Real Concern  FAVOR      0.536709
                                  NONE       0.425316
                                  AGAINST    0.037975
Feminist Movement                 AGAINST    0.493976
                                  FAVOR      0.316265
                                  NONE       0.189759
Hillary Clinton                   AGAINST    0.570392
                                  NONE       0.258345
                                  FAVOR      0.171263
Legalization of Abortion          AGAINST    0.543645
                                  NONE       0.271057
                                  FAVOR      0.185299
Name: proportion, dtype: float64

In [7]:
df_test.groupby(['Target'])['Stance'].value_counts(normalize=True)

Target                            Stance 
Atheism                           AGAINST    0.727273
                                  FAVOR      0.145455
                                  NONE       0.127273
Climate Change is a Real Concern  FAVOR      0.727811
                                  NONE       0.207101
                                  AGAINST    0.065089
Donald Trump                      AGAINST    0.422914
                                  NONE       0.367751
                                  FAVOR      0.209335
Feminist Movement                 AGAINST    0.642105
                                  FAVOR      0.203509
                                  NONE       0.154386
Hillary Clinton                   AGAINST    0.583051
                                  NONE       0.264407
                                  FAVOR      0.152542
Legalization of Abortion          AGAINST    0.675000
                                  FAVOR      0.164286
                                  NONE  

In [8]:
train_size = df_train.shape[0]
test_size = df_test.shape[0]
print(f'test proportion {test_size / (train_size+test_size)}')

test proportion 0.4016427104722793


## Create semeval-2016-raw dataset and store on Huggingface

In [9]:
def csv_to_hf_dataset(
    csv_paths: Union[str, List[str]], 
    split_names: Optional[List[str]] = None, 
    columns: Optional[List[str]] = None
) -> DatasetDict:
    """
    Convert CSV file(s) to a Hugging Face DatasetDict.
    
    Args:
        csv_paths (str or List[str]): Path to a single CSV file or list of CSV file paths
        split_names (List[str], optional): Names for each dataset split. 
                                           Defaults to ['train'] if single file, 
                                           or derived from filenames if multiple files.
        columns (List[str], optional): Specific columns to select. 
                                       If None, all columns are used.
    
    Returns:
        Hugging Face DatasetDict (even for single splits)
    """
    # Ensure csv_paths is a list
    if isinstance(csv_paths, str):
        csv_paths = [csv_paths]
    
    # Prepare split names if not provided
    if split_names is None:
        if len(csv_paths) == 1:
            split_names = ['train']
        else:
            # Generate split names from filenames
            split_names = [
                os.path.splitext(os.path.basename(path))[0] 
                for path in csv_paths
            ]
    
    # Validate input
    if len(csv_paths) != len(split_names):
        raise ValueError("Number of CSV paths must match number of split names")
    
    # Load datasets
    datasets = {}
    for path, split_name in zip(csv_paths, split_names):
        # Read CSV
        df = pd.read_csv(path)
        
        # Select specific columns if provided
        if columns:
            df = df[columns]
        
        # Convert to Hugging Face Dataset
        dataset = Dataset.from_pandas(df)
        datasets[split_name] = dataset
    
    # Always return DatasetDict
    return DatasetDict(datasets)

In [10]:
ds = csv_to_hf_dataset(csv_paths=[train, test],
                       split_names=['train', 'test']
                      )

In [11]:
# randomly shuffle data
ds = ds.shuffle(seed=42)
ds = ds.flatten_indices()

Flattening the indices: 100%|█████| 1956/1956 [00:00<00:00, 87666.12 examples/s]


In [12]:
ds.push_to_hub(repo_id='FundSciImpact/semeval-2016-raw', private=False)

Uploading the dataset shards: 100%|███████████████| 1/1 [00:00<00:00,  1.21it/s]


CommitInfo(commit_url='https://huggingface.co/datasets/FundSciImpact/semeval-2016-raw/commit/5b52c966795522ec26ca04d5f6263df677e6f930', commit_message='Upload dataset', commit_description='', oid='5b52c966795522ec26ca04d5f6263df677e6f930', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/FundSciImpact/semeval-2016-raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='FundSciImpact/semeval-2016-raw'), pr_revision=None, pr_num=None)

## Relabel Sentiment

In [12]:
for split in ds:
    print(f"Split {split}: {ds[split].unique('Sentiment')}")

Split train: ['neg', 'pos', 'other']
Split test: ['neg', 'pos', 'other']


In [13]:
ds_relabel = ds.map(lambda x: {'Sentiment': 'Yes' if x['Sentiment'] == 'pos' else 'No'})

Map: 100%|████████████████████████| 1956/1956 [00:00<00:00, 41069.17 examples/s]


In [14]:
for split in ds_relabel:
    print(f"Split {split}: {ds_relabel[split].unique('Sentiment')}")

Split train: ['No', 'Yes']
Split test: ['No', 'Yes']


In [15]:
ds_relabel.push_to_hub(repo_id='FundSciImpact/semeval-2016-sent-relabel', private=False)

Uploading the dataset shards: 100%|███████████████| 1/1 [00:01<00:00,  1.32s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/FundSciImpact/semeval-2016-sent-relabel/commit/21038f2a99315ae8c0c71ac2c5d03454bb15f0f0', commit_message='Upload dataset', commit_description='', oid='21038f2a99315ae8c0c71ac2c5d03454bb15f0f0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/FundSciImpact/semeval-2016-sent-relabel', endpoint='https://huggingface.co', repo_type='dataset', repo_id='FundSciImpact/semeval-2016-sent-relabel'), pr_revision=None, pr_num=None)

## Manipulate the data to get it into format used by 2404.12171

In [ ]:
dfs = [df_train, df_test]
for df in dfs:
    df['Stance_orig'] = df['Stance']
    df['Stance'] = df['Stance'].map({"FAVOR": "support", "NONE": "none", "AGAINST": "against"})

In [ ]:
ds_train = datasets.Dataset.from_pandas(df_train, preserve_index=False)
ds_test = datasets.Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
ds_train = ds_train.shuffle(seed=42)
ds_train = ds_train.flatten_indices()

ds_test = ds_test.shuffle(seed=42)
ds_test = ds_test.flatten_indices()

In [ ]:
# feat_stance = datasets.ClassLabel(num_classes=3, names=["against", "none", "support"])
# ds_train = ds_train.cast_column("Stance_int", feat_stance)
# ds_test = ds_test.cast_column("Stance_int", feat_stance)

In [ ]:
ds = datasets.DatasetDict({'train': ds_train, 'test': ds_test})

## Prompts

### Base prompt from 2404.12171

In [ ]:
prompt_base="""You are a helpful, respectful, and honest assistant for stance detection for a given target. 
Always answer from the possible options given below as helpfully as possible. 
Stance detection is the process of determining whether the author of a tweet is in support of or against a given target. 
The target may not always be explicitly mentioned in the text, and the tweet’s stance can be conveyed implicitly through subtext, regional and cultural references, or other implicit meanings.
The possible stances are:
* support: The tweet has a positive or supportive attitude towards the target, either explicitly or implicitly.
* against: The tweet opposes or criticizes the target, either explicitly or implicitly.
* none: The tweet is neutral or doesn’t have a stance towards the target.
"""

In [ ]:
print(prompt_base)

### Variation on base

In [ ]:
prompt_1="""You are a helpful, respectful, and honest assistant for stance detection for a given target. 
Always answer from the possible options given below as helpfully as possible. 
Stance detection is the process of determining whether the author of a tweet is in support of or against a given target. 
The target may not always be explicitly mentioned in the text, and the tweet’s stance can be conveyed implicitly through subtext, regional and cultural references, or other implicit meanings.
The possible stances are:
* support: The tweet has a positive or supportive attitude towards the target, either explicitly or implicitly.
* against: The tweet opposes or criticizes the target, either explicitly or implicitly.
* none: The tweet is neutral or doesn’t have a stance towards the target.
Your answer should contain only a single word taken from the list [support, against, none]."""

### Another variation on base

In [ ]:
prompt_2="""You are a helpful, respectful, and honest assistant for stance detection for a given target. 
Always answer from the possible options given below as helpfully as possible. 
Stance detection is the process of determining whether the author of a tweet is in support of or against a given target. 
The target may not always be explicitly mentioned in the text, and the tweet’s stance can be conveyed implicitly through subtext, regional and cultural references, or other implicit meanings.
The possible stances are:
* support: The tweet has a positive or supportive attitude towards the target, either explicitly or implicitly.
* against: The tweet opposes or criticizes the target, either explicitly or implicitly.
* none: The tweet is neutral or doesn’t have a stance towards the target.
Your answer must contain only a single word taken from the list [support, against, none]."""

### Base + simple Chain of Thought prompt

Added: "Lets think about this step by step"

## Applying formatting

In [ ]:
prompts = [prompt_base, prompt_1, prompt_2]

In [ ]:
def system(content: str):
    return { "role": "system", "content": content }

def assistant(content: str):
    return { "role": "assistant", "content": content }

def user(content: str):
    return { "role": "user", "content": content }

In [ ]:
def format_data(examples, prompt, column_name_suffix=None, eval_set=False):
    tweets = examples["Tweet"]
    targets = examples["Target"]
    labels = examples["Stance"]
    
    if column_name_suffix is None:
        column_name_chat = 'messages'
        column_name = 'text'
    else:
        column_name_chat = 'messages' + column_name_suffix
        column_name = 'text' + column_name_suffix
    
    texts = []
    chat_texts = []
    for tweet, target, label in zip(tweets, targets, labels):
        user_content = f"Tweet: {tweet} \n Stance towards the target {target}:"
        if not eval_set:
            # training dataset so add labels
            message = [system(prompt), user(user_content), assistant(label)]
            text = prompt + " " + user_content + " " + label
        else:
            message = [system(prompt), user(user_content), assistant("")]
            text = prompt + " " + user_content + " "
        texts.append(text)
        chat_texts.append(message)
    return { column_name : texts, column_name_chat: chat_texts}

def format_data_one_shot(examples, prompt, column_name_suffix=None, eval_set=False):
    tweets = examples["Tweet"]
    targets = examples["Target"]
    labels = examples["Stance"]
    
    if column_name_suffix is None:
        column_name_chat = 'messages'
        column_name = 'text'
    else:
        column_name_chat = 'messages' + column_name_suffix
        column_name = 'text' + column_name_suffix
    
    texts = []
    chat_texts = []
    one_shot = f"""Tweet: You cant think by yourself about life and believe in god. It just doesn’t add up #SemST \n Stance towards the target Atheism:"""
    for tweet, target, label in zip(tweets, targets, labels):
        user_content = f"Tweet: {tweet} \n Stance towards the target {target}:"
        if not eval_set:
            # training dataset so add labels
            message = [system(prompt), user(one_shot), assistant("support"), user(user_content), assistant(label)]
            text = prompt + one_shot + " support \n" + user_content + " " + label
        else:
            message = [system(prompt), user(one_shot), assistant("support"), user(user_content), assistant("")]
            text = prompt + one_shot + " support \n" + user_content + " "
        texts.append(text)
        chat_texts.append(message)
    return { column_name : texts, column_name_chat: chat_texts}

def format_data_cot(examples, prompt, column_name_suffix=None, eval_set=False):
    tweets = examples["Tweet"]
    targets = examples["Target"]
    labels = examples["Stance"]
    
    if column_name_suffix is None:
        column_name_chat = 'messages'
        column_name = 'text'
    else:
        column_name_chat = 'messages' + column_name_suffix
        column_name = 'text' + column_name_suffix
    
    texts = []
    chat_texts = []
    for tweet, target, label in zip(tweets, targets, labels):
        user_content = f"Tweet: {tweet} \n Stance towards the target {target}: Lets think about this step by step."
        if not eval_set:
            # training dataset so add labels
            message = [system(prompt), user(user_content), assistant(label)]
            text = prompt + " " + user_content + " " + label
        else:
            message = [system(prompt), user(user_content), assistant("")]
            text = prompt + " " + user_content + " "
        texts.append(text)
        chat_texts.append(message)
    return { column_name : texts, column_name_chat: chat_texts}

In [ ]:
for dataset_ in ds:
    bool_flag = False if dataset_ == 'train' else True
    for idx, prompt in enumerate(prompts):
        ds[dataset_] = ds[dataset_].map(format_data,
                                        batched=True,
                                        fn_kwargs={"prompt": prompt,
                                                   "column_name_suffix": f'_{idx}',
                                                   "eval_set": bool_flag},
                                        desc=f"Applying format_data with prompt {idx}"
                                        )
    ds[dataset_] = ds[dataset_].map(format_data_one_shot,
                                batched=True,
                                fn_kwargs={"prompt": prompt_base,
                                           "column_name_suffix": '_one_shot',
                                           "eval_set": bool_flag},
                                desc=f"Applying format_data_one_shot"
                                )
    ds[dataset_] = ds[dataset_].map(format_data_cot,
                                batched=True,
                                fn_kwargs={"prompt": prompt_base,
                                           "column_name_suffix": '_cot',
                                           "eval_set": bool_flag},
                                desc=f"Applying format_data_cot"
                                )

In [ ]:
ds.push_to_hub("FundSciImpact/semeval-2016", private=False)

# Dataset experimentation

In [ ]:
from datasets import load_dataset
ds = load_dataset('FundSciImpact/semeval-2016')

In [ ]:
label = ds['train'].features['Stance_int']
label

In [ ]:
label.names

In [ ]:
label.str2int('against'), label.str2int('none'), label.str2int('support')

In [ ]:
label.int2str(0), label.int2str(1), label.int2str(2)

In [ ]:
ds['train'].features['Stance']

In [ ]:
ds['train']['Stance'][0:4]

In [ ]:
ds['train'] = ds['train'].map(lambda x: {"Stance_conv": label.str2int(x['Stance'])},
                              batched=True,
                              desc="Converting labels")

In [ ]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub
user_secrets = UserSecretsClient()
hf_login = user_secrets.get_secret("HFapi")
huggingface_hub.login(token=hf_login)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')

In [ ]:
tokenizer.encode('against'), tokenizer.encode('none'), tokenizer.encode('support')

In [ ]:
ds['train']["Stance_conv"][0:4]